<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/brewly_logo.png"/>

# Practice SQL queries on Brewly ☕️

You work as a junior data analyst at **Brewly**, a small European coffee subscription company. The sales team needs answers about customers and orders, and they expect them today. The data lives in a small SQLite database with two tables: `customers` and `orders`. Your job is to write the SQL queries that pull the right numbers from these tables.

## A quick note on `sqlite3` and the cursor

Before you start, you need to understand two small things: what `sqlite3` is, and what a cursor does. You will use both in every task.

**What is `sqlite3`?** SQLite is a small database that lives inside a single file on your computer. The `sqlite3` module comes with Python. This is why teams use SQLite for tests, small tools, and local analysis work. 

**What is a connection?** A connection is the open line between your Python code and the database file. 

**What is a cursor?**
A cursor is the tool you use to run a SQL query through the connection. Think of it as a pointer that walks through the result rows for you. You create it once with `conn.cursor()`. After that, you use it for every query:

1. `cursor.execute(query)` sends your SQL query to the database.
2. `cursor.fetchone()` returns one row of the result.
3. `cursor.fetchall()` returns every row of the result as a list.

Imagine a tiny library database `library.db` (download it from [here](https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/library.db)) with one table called `books`. Each row holds a `title` and an `author`. Here is the full flow from start to end:


In [68]:
import sqlite3

# 1. Open a connection to a database file.
conn = sqlite3.connect("src/library.db")

# 2. Create a cursor.
cursor = conn.cursor()

# 3. Write your SQL query as a string.
query = "SELECT title FROM books WHERE author = 'Jane Austen';"

# 4. Send the query to the database with execute().
cursor.execute(query)

# 5. Read the result back with fetchall().
rows = cursor.fetchall()

# 6. Print each row.
for row in rows:
    print(row)

# 7. Close the cursor and the connection when you finish.
cursor.close()
conn.close()

('Pride and Prejudice',)
('Emma',)
('Persuasion',)


## Setup: prepare the database

Run the two cells below once. The first cell writes the CSV files to your working folder. The second cell loads them into a SQLite database called `brewly.db` and opens a cursor. After the setup, you are ready to query.

In [69]:
customers_csv = """customer_id,name,country,signup_date
1,Marie Laurent,France,2024-01-15
2,Tom Schneider,Germany,2024-02-03
3,Sofia Rossi,Italy,2024-02-20
4,Liam OConnor,Ireland,2024-03-11
5,Anna Kowalski,Poland,2024-04-02
6,Pierre Dubois,France,2024-05-18
7,Elena Garcia,Spain,2024-06-07
8,Karl Weber,Germany,2024-07-22
"""

orders_csv = """order_id,customer_id,order_date,amount_eur,product
101,1,2024-08-01,42.50,Espresso Blend
102,2,2024-08-03,68.00,Filter Box
103,1,2024-08-15,42.50,Espresso Blend
104,3,2024-08-20,55.00,Decaf Pack
105,4,2024-09-02,30.00,Single Origin
106,2,2024-09-10,68.00,Filter Box
107,6,2024-09-18,120.00,Premium Box
108,7,2024-10-01,42.50,Espresso Blend
109,1,2024-10-05,55.00,Decaf Pack
110,8,2024-10-12,30.00,Single Origin
111,6,2024-10-20,68.00,Filter Box
"""

customers_file = open("customers.csv", "w")
customers_file.write(customers_csv)
customers_file.close()

orders_file = open("orders.csv", "w")
orders_file.write(orders_csv)
orders_file.close()

print("CSV files created.")

CSV files created.


In [72]:
import pandas as pd
import sqlite3

customers_df = pd.read_csv("customers.csv")
orders_df = pd.read_csv("orders.csv")

conn = sqlite3.connect("brewly.db")

customers_df.to_sql("customers", conn, if_exists="replace", index=False)
orders_df.to_sql("orders", conn, if_exists="replace", index=False)

cursor = conn.cursor()

print("Database ready. You can now write SQL queries.")

Database ready. You can now write SQL queries.


## Task 1: Check that the tables are loaded

Before you write any business query, you confirm the database holds the right data. The sales team gave you two files, and you want to make sure both tables exist and contain rows. A query that returns zero rows often means you queried the wrong table or the data did not load.

Write one SQL query that counts the rows in the `customers` table. Pass it to `cursor.execute()`. Call `cursor.fetchone()` to read the single number back. Print the result. Then do the same for the `orders` table with a second query.

You expect to see 8 customers and 11 orders. If you see different numbers, something went wrong during setup.

In [73]:
# Step 1: count rows in the customers table
query = "select count(*) from customers"
cursor.execute(query)
rows_cust = cursor.fetchone()
print(rows_cust)

# Step 2: count rows in the orders table.
query = "select count(*) from orders"
cursor.execute(query)
rows_orders = cursor.fetchone()
print(rows_orders)




(8,)
(11,)


## Task 2: List every customer from Germany

The sales lead in Berlin asks for a quick list of all German customers in the system. She wants every column from the `customers` table so she can see signup dates and IDs at the same time.

Write one SQL query that selects all columns from the `customers` table and keeps only the rows where the country is Germany. Pass it to `cursor.execute()`. Call `cursor.fetchall()` to get every matching row. Loop through the result and print each row on its own line.

You expect to see two customers: Tom Schneider and Karl Weber.

In [74]:
query = "select * from customers WHERE country like 'Germany'"
cursor.execute(query)
rows_germancusts = cursor.fetchall()
for row in rows_germancusts:
    print(row)

(2, 'Tom Schneider', 'Germany', '2024-02-03')
(8, 'Karl Weber', 'Germany', '2024-07-22')


## Task 3: Find recent customers

Marketing wants to send a welcome offer to customers who joined recently. They define recent as signed up after March 1, 2024. They do not need every column, only the name and the signup date for their email tool.

Write one SQL query that selects the `name` and `signup_date` columns from the `customers` table and keeps only the rows where the signup date is later than '2024-03-01'. Pass it to `cursor.execute()`. Fetch all rows with `cursor.fetchall()`. Print each row in a loop.

You expect to see five customers in the result.

In [75]:
query = "SELECT name, signup_date FROM customers WHERE signup_date > '2024-03-01'"
cursor.execute(query)
rowsSignedOK = cursor.fetchall()
for row in rowsSignedOK:
    print(row)

('Liam OConnor', '2024-03-11')
('Anna Kowalski', '2024-04-02')
('Pierre Dubois', '2024-05-18')
('Elena Garcia', '2024-06-07')
('Karl Weber', '2024-07-22')


## Task 4: Filter and sort big orders

Finance wants to review every order above 50 euros. They want to start with the biggest order at the top because they suspect one large refund came in last month. They need the order ID, the amount, and the product name.

Write one SQL query that selects `order_id`, `amount_eur`, and `product` from the `orders` table, keeps only the rows where the amount is greater than 50, and sorts the result by amount from the highest to the lowest. Pass it to `cursor.execute()`. Fetch all rows with `cursor.fetchall()`. Print each row in a loop.

You expect the Premium Box order at 120 euros to appear first.

In [76]:
query = "SELECT order_id, amount_eur as a, product FROM orders WHERE amount_eur > 50 ORDER BY a DESC"
cursor.execute(query)
rowsSignedOK = cursor.fetchall()
for row in rowsSignedOK:
    print(row)

(107, 120.0, 'Premium Box')
(102, 68.0, 'Filter Box')
(106, 68.0, 'Filter Box')
(111, 68.0, 'Filter Box')
(104, 55.0, 'Decaf Pack')
(109, 55.0, 'Decaf Pack')


## Task 5: Total revenue per product

The product manager wants to know which product brings in the most money. She has no time to add things by hand. She wants one row per product and the total revenue next to it, sorted with the best seller on top.

Write one SQL query that selects the `product` column and the sum of `amount_eur` from the `orders` table. Give the sum the alias `total_revenue` so the column has a clear name in the result. Group the rows by product. Sort the result by the total revenue from the highest to the lowest. Pass it to `cursor.execute()`. Fetch all rows with `cursor.fetchall()`. Print each row in a loop.

You expect five rows, one for each distinct product.

In [77]:
query = """
SELECT product,
SUM(amount_eur) AS total_revenue
FROM orders
GROUP BY product
ORDER BY total_revenue DESC
"""

cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)




('Filter Box', 204.0)
('Espresso Blend', 127.5)
('Premium Box', 120.0)
('Decaf Pack', 110.0)
('Single Origin', 60.0)


## Task 6: Join customers and orders

The sales lead wants to see who placed each order. The `orders` table only stores `customer_id`, which is not useful on its own. To attach the customer name and country, you need to join the two tables on `customer_id`. The team also wants to see the orders in the order they came in, oldest first.

Write one SQL query that returns the customer name, the customer country, the order date, and the order amount. Use an `INNER JOIN` between `customers` and `orders` on the `customer_id` column. Sort the result by `order_date` in ascending order. Pass it to `cursor.execute()`. Fetch all rows with `cursor.fetchall()`. Print each row in a loop.

You expect 11 rows in the result, one per order. Short aliases like `c` for `customers` and `o` for `orders` keep the query easy to read.

In [78]:
query = """
SELECT c.name, c.country, o.order_date, o.amount_eur
FROM customers AS c
JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.order_date
"""

cursor.execute(query)
rows = cursor.fetchall()
for row in rows:
    print(row)

('Marie Laurent', 'France', '2024-08-01', 42.5)
('Tom Schneider', 'Germany', '2024-08-03', 68.0)
('Marie Laurent', 'France', '2024-08-15', 42.5)
('Sofia Rossi', 'Italy', '2024-08-20', 55.0)
('Liam OConnor', 'Ireland', '2024-09-02', 30.0)
('Tom Schneider', 'Germany', '2024-09-10', 68.0)
('Pierre Dubois', 'France', '2024-09-18', 120.0)
('Elena Garcia', 'Spain', '2024-10-01', 42.5)
('Marie Laurent', 'France', '2024-10-05', 55.0)
('Karl Weber', 'Germany', '2024-10-12', 30.0)
('Pierre Dubois', 'France', '2024-10-20', 68.0)


## Task 7: Find the top three customers

The CEO asks one question before her board meeting: who are the three customers who spent the most money so far? She wants the name and the total amount spent for each one. She has five minutes.

Write one SQL query that joins `customers` and `orders` on `customer_id`, sums the `amount_eur` for each customer, and gives the sum the alias `total_spent`. Group the rows by customer name. Sort the result by `total_spent` from the highest to the lowest. Use `LIMIT` to keep only the top three rows. Pass the query to `cursor.execute()`. Fetch the rows with `cursor.fetchall()`. Print each row in a loop.

You expect exactly three rows in the result.

In [80]:
query = """
SELECT c.name, SUM(o.amount_eur) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.name
ORDER BY total_spent DESC
LIMIT 3
"""
cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)


('Pierre Dubois', 188.0)
('Marie Laurent', 140.0)
('Tom Schneider', 136.0)


## Task 8: Read a SQL result back into a DataFrame

The CEO loved the top three list. Now she wants the same data as a clean table she can paste into her slide. A DataFrame prints with column names and aligned columns, which is what she needs.

Reuse the SQL query you wrote in Task 7. Pass it to `pd.read_sql_query()` together with the `conn` connection object. Store the returned DataFrame in a variable. Print the DataFrame.

You expect the same three customers as in Task 7, but displayed as a DataFrame with proper column headers.

In [81]:
df = pd.read_sql_query(query, conn)
df.head()

,name,total_spent
0,Pierre Dubois,188.0
1,Marie Laurent,140.0
2,Tom Schneider,136.0


## Close the cursor and the connection

When you finish working with a database, you close the cursor and the connection. This releases the database file and prevents resource leaks. Forgetting this step is a common mistake in production code.

In [83]:
cursor.close()
conn.close()
print("Cursor and connection closed.")

ProgrammingError: Cannot operate on a closed database.